# ComplianceGPT — Pipeline Acceptance Tests

This notebook is a **regression gate** for the ComplianceGPT pipeline.

It checks:
- **Final Answer Contract schema** (required fields exist)
- **Status ↔ evidence consistency invariants**
- Basic **PARAMS_REQUIRED** and **NO_EVIDENCE** behavioral constraints (non-brittle)

It does **not** measure retrieval quality. Use batch evaluation notebooks for that.


In [19]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import os, warnings

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")


In [21]:
import os, sys

PROJECT_ROOT = "/content/drive/MyDrive/ComplianceGPT"
assert os.path.isdir(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import compliancegpt
print("compliancegpt.__file__ =", compliancegpt.__file__)


compliancegpt.__file__ = None


In [22]:
# --- Canonical paths (Drive) ---
import os

DATA_DIR = os.path.join(PROJECT_ROOT, "data")

REV5_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev5_catalog.jsonl")
REV4_CATALOG_PATH = os.path.join(DATA_DIR, "ccs", "nist800-53", "NIST_SP-800-53_rev4_catalog.jsonl")

REV5_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev5_gold-set_100q.csv")
REV4_GOLD_PATH = os.path.join(DATA_DIR, "gold_standard_datasets", "nist800-53", "nist_sp800-53_rev4_gold-set_36q.csv")

ORG_PROFILE_REV5 = os.path.join(DATA_DIR, "ODP", "rev5", "org_profile_example_rev5.yaml")
ORG_PROFILE_REV4 = os.path.join(DATA_DIR, "ODP", "rev4", "org_profile_example_rev4.yaml")


In [23]:
# --- load gold sets ---
from typing import Any, Dict, Optional
import pandas as pd

def load_gold(framework_version: str) -> pd.DataFrame:
    path = REV5_GOLD_PATH if framework_version.lower() == "rev5" else REV4_GOLD_PATH
    # use latin1 for robustness.
    df = pd.read_csv(path, encoding="latin1")

    # Standardize columns to pipeline-friendly names
    if "id" in df.columns and "query_id" not in df.columns:
        df["query_id"] = df["id"].astype(str)
    if "question" in df.columns and "original_query" not in df.columns:
        df["original_query"] = df["question"].astype(str)
    return df

def get_gold_row(df: pd.DataFrame, qid: Any) -> Dict[str, Any]:
    qid_str = str(qid)
    m = df[df["query_id"].astype(str) == qid_str]
    if m.empty:
        raise KeyError(f"query_id not found in gold set: {qid_str}")
    return dict(m.iloc[0].to_dict())


In [24]:
# --- Build pipeline (single place to configure knobs) ---
from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline

def build_pipeline(
    *,
    framework_version: str,
    use_org_profile: bool,
    verify_strict_extras: bool = False,
    verify_strict_verbatim: bool = True,
    verify_strict_version: bool = False,
) -> ComplianceGPTPipeline:
    fw = framework_version.lower().strip()
    if fw not in {"rev4", "rev5"}:
        raise ValueError(f"framework_version must be rev4 or rev5 (got {framework_version!r})")

    ccs_path = REV5_CATALOG_PATH if fw == "rev5" else REV4_CATALOG_PATH

    org_profile = None
    if use_org_profile:
        org_profile = ORG_PROFILE_REV5 if fw == "rev5" else ORG_PROFILE_REV4
        if not os.path.exists(org_profile):
            print(f"[WARN] org_profile not found; continuing without profile: {org_profile}")
            org_profile = None

    pipe = ComplianceGPTPipeline(
        framework_version=fw,
        ccs_path=ccs_path,
        use_qur=False,                 # acceptance tests: keep deterministic
        org_profile_path=org_profile,  # optional
        verify_strict_extras=bool(verify_strict_extras),
        verify_strict_verbatim=bool(verify_strict_verbatim),
        verify_strict_version=bool(verify_strict_version),
    )
    return pipe


In [25]:
# === USER CONFIG ===
FRAMEWORK_VERSION = "rev5"   # "rev4" or "rev5"
USE_GENERATOR = True         # True exercises contract construction; False isolates retrieval

USE_ORG_PROFILE = False      # Off by default for acceptance tests

VERIFY_STRICT_EXTRAS = False
VERIFY_STRICT_VERBATIM = True
VERIFY_STRICT_VERSION = False

TOP_K = 12


In [26]:
# --- Config pipeline ---
pipe = build_pipeline(
    framework_version=FRAMEWORK_VERSION,
    use_org_profile=USE_ORG_PROFILE,
    verify_strict_extras=VERIFY_STRICT_EXTRAS,
    verify_strict_verbatim=VERIFY_STRICT_VERBATIM,
    verify_strict_version=VERIFY_STRICT_VERSION,
)
print("[OK] Pipeline initialized.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[Pipeline] 4-bit requested but bitsandbytes unavailable. Falling back to full precision. (No module named 'bitsandbytes')


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[OK] Pipeline initialized.


In [27]:
# --- Acceptance assertions (contract invariants) ---
from typing import List

def assert_contract_schema(c: Dict[str, Any]) -> None:
    required = [
        "answer_text",
        "evidence_spans",
        "status",
        "odp_required_list",
        "primary_citation",
        "all_citations",
        "answer_text_with_citation",
        "contract_mode",
    ]
    missing = [k for k in required if k not in c]
    assert not missing, f"Missing required contract fields: {missing}"
    assert c.get("contract_mode") == "provably_extractive", f"contract_mode mismatch: {c.get('contract_mode')}"
    assert isinstance(c.get("evidence_spans"), list), "evidence_spans must be a list"

def assert_status_consistency(c: Dict[str, Any]) -> None:
    status = str(c.get("status", "")).upper().strip()
    spans = c.get("evidence_spans") or []
    if status == "NO_EVIDENCE":
        assert len(spans) == 0, "NO_EVIDENCE must have empty evidence_spans"
    if status in {"OK", "PARAMS_REQUIRED"}:
        assert len(spans) >= 1, f"{status} must have at least 1 evidence span"

def run_one(query: str, gold_row: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    # IMPORTANT: verifier should only be requested when gold labels exist.
    run_verify = bool(gold_row is not None)

    out = pipe.answer(
        query,
        top_k=int(TOP_K),
        rewrites=None,
        gold_row=gold_row,
        use_generator=bool(USE_GENERATOR),
        run_verify=run_verify,
    )
    c = out.get("contract", out)

    assert_contract_schema(c)
    assert_status_consistency(c)
    return c


## Test set

In [28]:
# --- Select test cases from gold (stable IDs) ---
gold_df = load_gold(FRAMEWORK_VERSION)

# Test-1 (ODP-related): ID 1 is known to be ODP-related in the provided sets.
gold_row_params = get_gold_row(gold_df, 1)
query_params = str(gold_row_params["original_query"]).strip()
print("[T1] PARAMS_REQUIRED candidate query:", query_params)

# Test-3 (OK-ish): choose a simple early ID; no strong retrieval-quality expectation.
try:
    gold_row_ok = get_gold_row(gold_df, 3)
    query_ok = str(gold_row_ok["original_query"]).strip()
except Exception:
    gold_row_ok = None
    query_ok = "What is the principle of least privilege?"


[T1] PARAMS_REQUIRED candidate query: When must account managers and the designated personnel be notified about user account changes?


In [29]:
# --- 1) PARAMS_REQUIRED behavior (non-brittle) ---
c1 = run_one(query_params, gold_row=gold_row_params)

s1 = str(c1.get("status","")).upper().strip()
sp1 = c1.get("evidence_spans") or []
odps1 = c1.get("odp_required_list") or []

print(f"[T1] status={s1} evidence_spans_len={len(sp1)} odp_required_list_len={len(odps1)}")

# Hard requirement: if pipeline claims PARAMS_REQUIRED, it must list required params.
if s1 == "PARAMS_REQUIRED":
    assert len(odps1) >= 1, "PARAMS_REQUIRED must have non-empty odp_required_list"


[T1] status=PARAMS_REQUIRED evidence_spans_len=4 odp_required_list_len=4


In [30]:
# --- 2) NO_EVIDENCE behavior (nonsense query; consistency only) ---
nonsense_query = "ZZZ-NONEXISTENT-CONTROL-ID: this query should not match anything"
c2 = run_one(nonsense_query, gold_row=None)

s2 = str(c2.get("status","")).upper().strip()
sp2 = c2.get("evidence_spans") or []

print(f"[T2] status={s2} evidence_spans_len={len(sp2)} (NO_EVIDENCE is allowed but not guaranteed)")


[T2] status=PARAMS_REQUIRED evidence_spans_len=1 (NO_EVIDENCE is allowed but not guaranteed)


In [31]:
# --- 3) Schema stability on an OK-ish gold question ---
c3 = run_one(query_ok, gold_row=gold_row_ok)

s3 = str(c3.get("status","")).upper().strip()
sp3 = c3.get("evidence_spans") or []
odps3 = c3.get("odp_required_list") or []

print(f"[T3] status={s3} evidence_spans_len={len(sp3)} odp_required_list_len={len(odps3)}")

# --- Summary ---
summary = {
    "T1_status": s1,
    "T2_status": s2,
    "T3_status": s3,
    "use_generator": bool(USE_GENERATOR),
    "framework_version": str(FRAMEWORK_VERSION),
}
print("\n=== Acceptance Test Summary ===")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\n[OK] Acceptance tests completed.")


[T3] status=OK evidence_spans_len=2 odp_required_list_len=0

=== Acceptance Test Summary ===
T1_status: PARAMS_REQUIRED
T2_status: PARAMS_REQUIRED
T3_status: OK
use_generator: True
framework_version: rev5

[OK] Acceptance tests completed.


Mini batch run

In [32]:
print(gold_df.columns.tolist())
print(gold_df["query_id"].head(15).tolist())
print(gold_df["query_id"].dtype)

['id', 'question', 'gold_source_doc', 'gold_source_version', 'control_id', 'answer', 'gold_exact_citation', 'gold_control_path', 'odp_required', 'resolution_policy', 'source_url', 'effective_date', 'query_id', 'original_query']
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15']
object


In [33]:
import pandas as pd

# Pick a tiny smoke set from already-loaded gold DataFrame.

# Use the already loaded gold_df DataFrame instead of the path string
mini_df = gold_df.head(5).copy()

rows = []
for _, r in mini_df.iterrows():
    q = str(r["question"])
    out = run_one(q, gold_row=r)

    rows.append({
        "query_id": r.get("query_id", ""),
        "question": q[:120],
        "status": out.get("status", ""),
        "evidence_spans_len": len(out.get("evidence_spans") or []),
        "odp_required_list_len": len(out.get("odp_required_list") or []),
        "verifier_pass": out.get("verifier_pass", None),
        "verifier_errors": "; ".join(out.get("verifier_errors") or []),
        "primary_citation": out.get("primary_citation", ""),
    })

mini_results = pd.DataFrame(rows)
display(mini_results)

,query_id,question,status,evidence_spans_len,odp_required_list_len,verifier_pass,verifier_errors,primary_citation
0,1,When must account managers and the designated ...,PARAMS_REQUIRED,4,4,True,,ac-2_smt.h
1,2,What must an organization enforce regarding lo...,OK,1,0,True,,ac-3_smt
2,3,What principle should be employed to limit use...,OK,2,0,True,,ac-6_smt
3,4,How must a system be configured to handle cons...,PARAMS_REQUIRED,4,3,True,,ac-7_gdn
4,5,What actions must a system take to lock a sess...,PARAMS_REQUIRED,3,1,True,,ac-11_gdn


In [34]:
# Example mixed smoke set:
# - 3 from rev5 gold
# - 2 more rev5 rows that are ODP-heavy if you know them
# - later repeat same idea for rev4 and error bank

mini_df = pd.concat([
    gold_df.head(3),
    gold_df.iloc[10:12],   # replace with better ODP rows if you know them
], ignore_index=True)

rows = []
for _, r in mini_df.iterrows():
    q = str(r["question"])
    out = run_one(q, gold_row=r)

    rows.append({
        "query_id": r.get("query_id", ""),
        "status": out.get("status", ""),
        "evidence_spans_len": len(out.get("evidence_spans") or []),
        "odp_required_list_len": len(out.get("odp_required_list") or []),
        "verifier_pass": out.get("verifier_pass", None),
        "verifier_errors": "; ".join(out.get("verifier_errors") or []),
        "primary_citation": out.get("primary_citation", ""),
    })

mini_results = pd.DataFrame(rows)
display(mini_results)

,query_id,status,evidence_spans_len,odp_required_list_len,verifier_pass,verifier_errors,primary_citation
0,1,PARAMS_REQUIRED,4,4,True,,ac-2_smt.h
1,2,OK,1,0,True,,ac-3_smt
2,3,OK,2,0,True,,ac-6_smt
3,11,PARAMS_REQUIRED,5,3,True,,au-2_smt.d
4,12,OK,6,0,False,MissedDocIds:['au-3_smt'],au-3_smt.a


In [35]:
mask = gold_df["query_id"].astype(str).str.strip() == "4"
row4 = gold_df[mask].iloc[0]
q4 = str(row4["question"])

out4 = run_one(q4, gold_row=row4)

print("=== QUERY_ID ===")
print(row4.get("query_id"))

print("\n=== QUERY ===")
print(q4)

print("\n=== STATUS ===")
print(out4.get("status"))

print("\n=== ODP REQUIRED LIST ===")
print(out4.get("odp_required_list"))

print("\n=== ANSWER TEXT ===")
print(out4.get("answer_text", "")[:2000])

print("\n=== EVIDENCE SPANS ===")
for s in out4.get("evidence_spans", []):
    print("-", s.get("source_id"))

print("\n=== VERIFIER PASS ===")
print(out4.get("verifier_pass"))

print("\n=== VERIFIER ERRORS ===")
print(out4.get("verifier_errors"))

print("\n=== DEBUG ===")
dbg = out4.get("debug", {}) or {}
for k in ["query_plan", "retrieval_meta", "allowed_controls", "doc_filter_mode_used", "winner_control", "selector_raw"]:
    print(f"\n--- {k} ---")
    print(dbg.get(k))

=== QUERY_ID ===
4

=== QUERY ===
How must a system be configured to handle consecutive invalid logon attempts?

=== STATUS ===
PARAMS_REQUIRED

=== ODP REQUIRED LIST ===
['ac-07_odp.01', 'ac-07_odp.02', 'ac-07_odp.03']

=== ANSWER TEXT ===
Enforce a limit of {{ insert: param, ac-07_odp.01 }} consecutive invalid logon attempts by a user during a {{ insert: param, ac-07_odp.02 }} ; and

Automatically {{ insert: param, ac-07_odp.03 }} when the maximum number of unsuccessful attempts is exceeded.

a. Enforce a limit of {{ insert: param, ac-07_odp.01 }} consecutive invalid logon attempts by a user during a {{ insert: param, ac-07_odp.02 }} ; and b. Automatically {{ insert: param, ac-07_odp.03 }} when the maximum number of unsuccessful attempts is exceeded.

The need to limit unsuccessful logon attempts and take subsequent action when the maximum number of attempts is exceeded applies regardless of whether the logon occurs via a local or network connection. Due to the potential for denial o

In [36]:
mask = gold_df["query_id"].astype(str).str.strip() == "4"
print("matched rows:", mask.sum())

row4_df = gold_df[mask]
display(row4_df.head())

row4 = row4_df.iloc[0]
q4 = str(row4["question"])
print(q4)

matched rows: 1


,id,question,gold_source_doc,gold_source_version,control_id,answer,gold_exact_citation,gold_control_path,odp_required,resolution_policy,source_url,effective_date,query_id,original_query
3,4,How must a system be configured to handle cons...,NIST SP 800-53,Revision 5,AC-7,"Enforce a limit of {{ insert: param, ac-7_odp_...","Enforce a limit of {{ insert: param, ac-07_odp...",ac-7_smt.a\nac-7_smt.b,ac-07_odp.01\nac-07_odp.02\nac-07_odp.03,FILL_FROM_PROFILE,https://github.com/usnistgov/oscal-content/blo...,9/23/20,4,How must a system be configured to handle cons...


How must a system be configured to handle consecutive invalid logon attempts?
